In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

100%|██████████| 346M/346M [01:26<00:00, 4.21MB/s] 

Extracting files...


Path to dataset files: C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2


####Step 1: Load and Inspect the Dataset Structure

In [8]:
import os
from pathlib import Path

def inspect_dataset_structure(dataset_path):
    dataset_path = Path(dataset_path)
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f"Dataset root: {dataset_path}")
    print(f"Number of classes: {len(classes)}")
    print(f"Classes found: {classes}")
    return classes

# Use the actual path from kagglehub
dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"
classes = inspect_dataset_structure(dataset_path)

Dataset root: C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2
Number of classes: 3
Classes found: ['seg_pred', 'seg_test', 'seg_train']


###Step 2: Analyze Class Distribution 

In [15]:
import os

dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"

print(f"Checking: {dataset_path}")
print(f"Exists: {os.path.exists(dataset_path)}\n")

def print_directory_tree(path, indent=0, max_depth=2):
    """Print directory structure"""
    if indent > max_depth:
        return
    
    try:
        items = os.listdir(path)
        for item in items[:10]:  # Show first 10 items
            item_path = os.path.join(path, item)
            print("  " * indent + f"📁 {item}" if os.path.isdir(item_path) else "  " * indent + f"📄 {item}")
            
            if os.path.isdir(item_path) and indent < max_depth:
                print_directory_tree(item_path, indent + 1, max_depth)
    except PermissionError:
        print("  " * indent + "🔒 Permission denied")
    except Exception as e:
        print("  " * indent + f"❌ Error: {e}")

print_directory_tree(dataset_path)

Checking: C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2
Exists: True

📁 seg_pred
  📁 seg_pred
    📄 10004.jpg
    📄 10005.jpg
    📄 10012.jpg
    📄 10013.jpg
    📄 10017.jpg
    📄 10021.jpg
    📄 1003.jpg
    📄 10034.jpg
    📄 10038.jpg
    📄 10040.jpg
📁 seg_test
  📁 seg_test
    📁 buildings
    📁 forest
    📁 glacier
    📁 mountain
    📁 sea
    📁 street
📁 seg_train
  📁 seg_train
    📁 buildings
    📁 forest
    📁 glacier
    📁 mountain
    📁 sea
    📁 street


###Step 3: Compute Image Statistics 

In [11]:
from PIL import Image
import numpy as np
import cv2
import os

# ✅ Use your actual dataset path
dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"

def compute_image_statistics(dataset_dir, sample_size=500):
    all_paths = []
    for root, _, files in os.walk(dataset_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(os.path.join(root, f))

    print(f"Total images found: {len(all_paths)}")  # ✅ sanity check

    sampled = np.random.choice(all_paths, min(sample_size, len(all_paths)), replace=False)
    widths, heights, means, stds = [], [], [], []

    for path in sampled:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)

        img_arr = cv2.imread(path)
        if img_arr is None:          # ✅ skip unreadable images
            continue
        img_arr = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        means.append(img_arr.mean(axis=(0, 1)))
        stds.append(img_arr.std(axis=(0, 1)))

    return {
        "mean_resolution": (np.mean(widths), np.mean(heights)),
        "std_resolution":  (np.std(widths),  np.std(heights)),
        "channel_mean":    np.mean(means, axis=0),
        "channel_std":     np.mean(stds,  axis=0)
    }

# ✅ Point to your actual path
stats = compute_image_statistics(dataset_path)
print(f"Mean resolution : {stats['mean_resolution']}")
print(f"Std  resolution : {stats['std_resolution']}")
print(f"Channel mean    : {stats['channel_mean']}")
print(f"Channel std     : {stats['channel_std']}")

Total images found: 24335
Mean resolution : (np.float64(150.0), np.float64(149.91))
Std  resolution : (np.float64(0.0), np.float64(1.8809306207300682))
Channel mean    : [0.43228498 0.45976186 0.45816216]
Channel std     : [0.23481975 0.23353215 0.23984998]


### Step 4: Run Quality Checks 

In [12]:
# ✅ Your actual dataset path
dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"

# ============================================
# FUNCTION 2 — Quality Checks
# ============================================
def run_quality_checks(dataset_dir, blur_threshold=100.0):
    corrupt, blurry = [], []
    for root, _, files in os.walk(dataset_dir):
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            path = os.path.join(root, f)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception as e:
                corrupt.append((path, str(e)))
                continue
            gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                score = cv2.Laplacian(gray, cv2.CV_64F).var()
                if score < blur_threshold:
                    blurry.append((path, score))
    return corrupt, blurry

# ============================================
# RUN BOTH — using correct dataset path
# ============================================
print("=" * 40)
print("Running image statistics...")
print("=" * 40)
stats = compute_image_statistics(dataset_path)
print(f"Mean resolution : {stats['mean_resolution']}")
print(f"Std  resolution : {stats['std_resolution']}")
print(f"Channel mean    : {stats['channel_mean']}")
print(f"Channel std     : {stats['channel_std']}")

print("\n" + "=" * 40)
print("Running quality checks...")
print("=" * 40)
corrupt, blurry = run_quality_checks(dataset_path)  # ✅ fixed path
print(f"Corrupt files : {len(corrupt)}")
print(f"Blurry images : {len(blurry)}")

Running image statistics...
Total images found: 24335
Mean resolution : (np.float64(150.0), np.float64(150.0))
Std  resolution : (np.float64(0.0), np.float64(0.0))
Channel mean    : [0.42886204 0.45621228 0.45357686]
Channel std     : [0.23336376 0.23426457 0.24436319]

Running quality checks...
Corrupt files : 0
Blurry images : 30


In [15]:
import os

dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"

print("Contents of dataset_path:")
for item in os.listdir(dataset_path):
    full_path = os.path.join(dataset_path, item)
    print(f"  {'[DIR]' if os.path.isdir(full_path) else '[FILE]'} {item}")

Contents of dataset_path:
  [DIR] seg_pred
  [DIR] seg_test
  [DIR] seg_train


In [16]:
def count_classes(dataset_dir):
    counts = {}

    # Check all subdirectories recursively
    for root, dirs, files in os.walk(dataset_dir):
        images = [f for f in files
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if images:  # ✅ only count folders that actually have images
            class_name = os.path.basename(root)
            counts[class_name] = len(images)

    return counts

# ============================================
# Run and debug counts
# ============================================
counts = count_classes(dataset_path)

print(f"Classes found: {len(counts)}")
for cls, count in counts.items():
    print(f"  {cls}: {count} images")

total_images = sum(counts.values())
print(f"\nTotal images: {total_images}")

# ✅ Safety check before running report
if len(counts) == 0:
    print("❌ No classes found! Check your dataset path.")
elif min(counts.values()) == 0:
    print("❌ Some classes have 0 images!")
else:
    print("\n✅ counts look good — running report...")
    report = produce_dataset_report(counts, stats, corrupt, blurry, total_images)

Classes found: 7
  seg_pred: 7301 images
  buildings: 2191 images
  forest: 2271 images
  glacier: 2404 images
  mountain: 2512 images
  sea: 2274 images
  street: 2382 images

Total images: 21335

✅ counts look good — running report...

       DATASET REPORT
  total_images        : 21335
  num_classes         : 7
  imbalance_ratio     : 3.33x
  mean_resolution     : 150x150
  channel_mean        : [0.4288620352745056, 0.45621228218078613, 0.45357686281204224]
  channel_std         : [0.23336376249790192, 0.2342645674943924, 0.2443631887435913]
  corrupt_files       : 0
  blurry_images       : 30 (0.1%)


In [17]:
import os
from collections import Counter

# ✅ Your actual dataset path
dataset_path = r"C:\Users\Hp\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2"

# ============================================
# STEP 1 — Count images per class
# ============================================
def count_classes(dataset_dir):
    counts = {}
    for class_name in os.listdir(dataset_dir):
        class_path = os.path.join(dataset_dir, class_name)
        if os.path.isdir(class_path):
            images = [f for f in os.listdir(class_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            counts[class_name] = len(images)
    return counts

# ============================================
# STEP 2 — Produce Dataset Report
# ============================================
def produce_dataset_report(counts, stats, corrupt, blurry, total_images):
    report = {
        "total_images"   : total_images,
        "num_classes"    : len(counts),
        "imbalance_ratio": f"{max(counts.values()) / min(counts.values()):.2f}x",
        "mean_resolution": f"{stats['mean_resolution'][0]:.0f}x{stats['mean_resolution'][1]:.0f}",
        "channel_mean"   : stats['channel_mean'].tolist(),
        "channel_std"    : stats['channel_std'].tolist(),
        "corrupt_files"  : len(corrupt),
        "blurry_images"  : f"{len(blurry)} ({100 * len(blurry) / total_images:.1f}%)"
    }

    print("\n" + "=" * 40)
    print("       DATASET REPORT")
    print("=" * 40)
    for key, value in report.items():
        print(f"  {key:<20}: {value}")
    print("=" * 40)

    return report

# ============================================
# STEP 3 — Collect all inputs and run report
# ============================================

# Count classes
counts = count_classes(dataset_path)
print("Class counts:")
for cls, count in counts.items():
    print(f"  {cls}: {count}")

# Total images
total_images = sum(counts.values())
print(f"\nTotal images: {total_images}")

# Run report using stats, corrupt, blurry from previous steps
report = produce_dataset_report(counts, stats, corrupt, blurry, total_images)

Class counts:
  seg_pred: 0
  seg_test: 0
  seg_train: 0

Total images: 0


ZeroDivisionError: division by zero